# 🎙️ BookVoice-AI Fine-tuning
**Perfekte emotionale Stimme für Rumi, Sufi & Gedichte**

**Runtime:** T4 GPU | Python 3

**Dauer:** ca. 2-4 Stunden

In [ ]:
#@title 1. Server & Projekt einrichten
import os, requests, zipfile, io, json
from pathlib import Path

SERVER_URL = "https://ahrar.aksoy-net.de"  #@param {type:"string"}
PROJECT_NAME = ""  #@param {type:"string"}

# API-URL normalisieren
if SERVER_URL.rstrip('/').endswith('/api'):
    API_BASE = SERVER_URL.rstrip('/')
else:
    API_BASE = SERVER_URL.rstrip('/') + '/api'

print(f"🌐 Server: {API_BASE}")
print(f"📁 Projekt: {PROJECT_NAME}")

r = requests.get(f"{API_BASE}/health", timeout=10)
r.raise_for_status()
print("✅ Verbindung erfolgreich")

In [ ]:
#@title 2. Dataset herunterladen
if not PROJECT_NAME:
    raise ValueError("❌ Bitte Projektnamen eingeben!")

dl_url = f"{API_BASE}/training/projects/{PROJECT_NAME}/export/download"
print(f"📥 Lade Dataset von {dl_url} ...")

resp = requests.get(dl_url)
if resp.status_code != 200:
    raise Exception(f"Download fehlgeschlagen (Status {resp.status_code})")

os.makedirs("/content/dataset", exist_ok=True)
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("/content/dataset")

print("✅ Dataset entpackt:")
!ls -la /content/dataset/

In [ ]:
#@title 3. Emotionstags automatisch hinzufügen
print("🏷️  Füge Style-Tags hinzu...")

sufi_words = ['allah', 'aşk', 'gönül', 'can', 'ruh', 'rumi', 'mesnevi', 'sufi', 'derviş', 'hakikat', 'mevlana']

def auto_tag(text):
    if any(w in text.lower() for w in sufi_words):
        return f"<sufi>{text}</sufi>"
    return f"<poem>{text}</poem>"

for csv_file in ["/content/dataset/metadata_train.csv", "/content/dataset/metadata_eval.csv"]:
    if not os.path.exists(csv_file):
        continue
    with open(csv_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    header = lines[0]
    new_lines = [header]
    for line in lines[1:]:
        if not line.strip():
            continue
        parts = line.strip().split('|')
        if len(parts) >= 2:
            parts[1] = auto_tag(parts[1])
        new_lines.append('|'.join(parts))
    with open(csv_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(new_lines))

print("✅ Tags hinzugefügt")
!head -3 /content/dataset/metadata_train.csv

In [ ]:
#@title 4. Coqui TTS installieren (ca. 3 Minuten)
print("⏳ Installiere Coqui TTS...")
!pip install TTS==0.22.0 -q
print("✅ Coqui TTS bereit")

In [ ]:
#@title 5. Basis-Modell laden (ca. 2 Minuten)
import os
os.environ["COQUI_TOS_AGREED"] = "1"
from TTS.api import TTS

print("⏳ Lade XTTS-v2 Basis-Modell...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=True)

# Modell-Pfad ermitteln
import glob
model_paths = glob.glob("/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2*")
if not model_paths:
    model_paths = glob.glob("/root/.local/share/tts/*xtts*")
MODEL_BASE_PATH = model_paths[0] if model_paths else "/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2"
print(f"✅ Modell bereit: {MODEL_BASE_PATH}")

In [ ]:
#@title 6. Fine-tuning starten (2-4 Stunden)
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from trainer import Trainer, TrainerArgs

print("⏳ Starte Fine-tuning...")
print("   fp32 Modus — wichtig für XTTS!")
print("   Dauer: ca. 2-4 Stunden")
print()

config = XttsConfig()
config.load_json(f"{MODEL_BASE_PATH}/config.json")

model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=MODEL_BASE_PATH)

trainer = Trainer(
    TrainerArgs(
        output_path="/content/finetuned",
        batch_size=2,
        grad_acumm=4,
        lr=5e-6,
        epochs=5,
        save_step=500,
        use_amp=False,
    ),
    config,
    model,
    train_samples="/content/dataset/metadata_train.csv",
    eval_samples="/content/dataset/metadata_eval.csv" if os.path.exists("/content/dataset/metadata_eval.csv") else None,
)

trainer.fit()
print("\n✅ Fine-tuning abgeschlossen!")

In [ ]:
#@title 7. Modell exportieren & hochladen
import shutil, zipfile

print("📦 Erstelle ZIP...")

os.makedirs("/content/final_model", exist_ok=True)

# Bestes Modell kopieren
best_model = sorted(Path("/content/finetuned").glob("**/*.pth"))
if best_model:
    shutil.copy(best_model[-1], "/content/final_model/model.pth")
    print(f"✅ Modell kopiert: {best_model[-1].name}")
else:
    raise Exception("❌ Kein Modell gefunden — Training abgeschlossen?")

# Config und Vocab kopieren
for fname in ["config.json", "vocab.json"]:
    src = f"{MODEL_BASE_PATH}/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/final_model/{fname}")
        print(f"✅ {fname} kopiert")

# ZIP erstellen
!zip -r /content/finetuned_model.zip /content/final_model

# Hochladen zu BookVoice
MODEL_NAME = f"{PROJECT_NAME}_finetuned"
print(f"📤 Lade Modell hoch als: {MODEL_NAME}")

with open("/content/finetuned_model.zip", "rb") as f:
    r = requests.post(
        f"{API_BASE}/training/models/upload",
        files={"file": f},
        data={"name": MODEL_NAME},
        timeout=300
    )

if r.status_code == 200:
    print(f"✅ Modell '{MODEL_NAME}' erfolgreich hochgeladen!")
    print()
    print("🎉 Fertig! Jetzt in BookVoice:")
    print("   1. Trainingsraum öffnen")
    print("   2. Karte '4 · Trainiertes Modell'")
    print(f"   3. '{MODEL_NAME}' → Aktivieren")
    print("   4. Studio → Hörbuch mit eigener Stimme! 🎙️")
else:
    print(f"❌ Upload fehlgeschlagen: {r.status_code}")
    print(r.text[:300])
    print("\n📥 Manueller Download:")
    from google.colab import files
    files.download("/content/finetuned_model.zip")